In [1]:

import torch
from torch import nn


class Decoder(nn.Module):
    def __init__(self, d_model, d_ff, n_heads, n_decoder_layers):
        super().__init__()
        self.layers = nn.ModuleList([DecoderLayer(d_model, d_ff, n_heads) for _ in range(n_decoder_layers)])

    def forward(self, decoder_inputs, mask=None):
        for layer in self.layers:
            decoder_inputs = layer(decoder_inputs, mask=mask)
        return decoder_inputs


class DecoderLayer(nn.Module):
    def __init__(self, d_model, d_ff, n_heads):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, d_model, d_model, n_heads)
        self.ffn = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, decoder_inputs, mask=None):
        attn1 = self.self_attn(q=decoder_inputs, mask=mask)
        x = self.norm1(decoder_inputs + attn1)

        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        return x


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear2(torch.relu(self.linear1(x)))


class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim: int, attn_dim: int, output_dim: int, num_heads: int):
        super().__init__()
        self.embed_dim = embed_dim
        self.attn_dim = attn_dim
        self.output_dim = output_dim
        self.num_heads = num_heads
        self.head_dim = attn_dim // num_heads

        self.q_proj = nn.Linear(embed_dim, self.attn_dim, bias=False)
        self.k_proj = nn.Linear(embed_dim, self.attn_dim, bias=False)
        self.v_proj = nn.Linear(embed_dim, self.attn_dim, bias=False)

        self.out_proj = nn.Linear(self.attn_dim, self.output_dim, bias=False)

    def forward(self, q, k=None, v=None, mask=None):
        if k is None: k = q
        if v is None: v = q

        batch_size, seq_len_q, _ = q.shape
        seq_len_k = k.shape[1]

        q = self.q_proj(q).view(batch_size, seq_len_q, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(k).view(batch_size, seq_len_k, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(v).view(batch_size, v.shape[1], self.num_heads, self.head_dim).transpose(1, 2)

        attn_score = torch.matmul(q, k.transpose(-2, -1))
        attn_score = attn_score / torch.sqrt(torch.tensor(self.head_dim, dtype=torch.float32))

        if mask is not None:
            mask = mask.unsqueeze(0).unsqueeze(1)
            attn_score = attn_score.masked_fill(mask == 0, -1e9)

        attn_weight = torch.softmax(attn_score, dim=-1)

        output = torch.matmul(attn_weight, v)
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len_q, self.attn_dim)
        return self.out_proj(output)

In [2]:
d_model = 128
d_ff = d_model * 2
n_heads = 2
n_decoder_layers = 4
max_seq_len = 5


class GPT(nn.Module):
    def __init__(self, vocab_size, max_seq_len):
        super().__init__()
        self.decoder = Decoder(d_model, d_ff, n_heads, n_decoder_layers)
        self.wte = nn.Embedding(vocab_size, d_model)
        self.wpe = nn.Embedding(max_seq_len, d_model)
        self.layer_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, inputs_ids, mask=None):
        batch_size, seq_len = inputs_ids.shape
        token = self.wte(inputs_ids)
        position = torch.arange(0, seq_len) % self.wpe.num_embeddings
        position = position.unsqueeze(0).repeat(batch_size, 1)
        position = self.wpe(position)
        embedding = token + position
        decoder_output = self.decoder(embedding, mask)
        decoder_output = self.layer_norm(decoder_output)
        return self.lm_head(decoder_output)


In [3]:
text = """
臣密言：臣以险衅，夙遭闵凶。
生孩六月，慈父见背；行年四岁，舅夺母志。
祖母刘愍臣孤弱，躬亲抚养。
臣少多疾病，九岁不行，零丁孤苦，至于成立。
既无伯叔，终鲜兄弟，门衰祚薄，晚有儿息。
外无期功强近之亲，内无应门五尺之僮，茕茕孑立，形影相吊。
而刘夙婴疾病，常在床蓐，臣侍汤药，未曾废离。
"""

words = set(text)
vocab_size = len(words)
word_to_index = {word: i for i, word in enumerate(words)}
index_to_word = {i: word for i, word in enumerate(words)}

print(word_to_index)

{'于': 0, '儿': 1, '茕': 2, '药': 3, '刘': 4, '志': 5, '四': 6, '夺': 7, '废': 8, '汤': 9, '疾': 10, '弱': 11, '\n': 12, '成': 13, '立': 14, '尺': 15, '行': 16, '祚': 17, '床': 18, '在': 19, '伯': 20, '亲': 21, '养': 22, '生': 23, '闵': 24, '僮': 25, '常': 26, '臣': 27, '未': 28, '密': 29, '险': 30, '父': 31, '凶': 32, '多': 33, '薄': 34, '形': 35, '晚': 36, '叔': 37, '以': 38, '功': 39, '外': 40, '丁': 41, '五': 42, '祖': 43, '；': 44, '至': 45, '鲜': 46, '不': 47, '无': 48, '遭': 49, '年': 50, '月': 51, '苦': 52, '愍': 53, '侍': 54, '零': 55, '期': 56, '内': 57, '见': 58, '有': 59, '吊': 60, '六': 61, '而': 62, '九': 63, '母': 64, '孩': 65, '。': 66, '弟': 67, '曾': 68, '近': 69, '：': 70, '病': 71, '强': 72, '慈': 73, '兄': 74, '衰': 75, '，': 76, '夙': 77, '岁': 78, '终': 79, '息': 80, '孤': 81, '言': 82, '抚': 83, '应': 84, '孑': 85, '相': 86, '背': 87, '少': 88, '舅': 89, '影': 90, '离': 91, '婴': 92, '既': 93, '躬': 94, '门': 95, '蓐': 96, '之': 97, '衅': 98}


In [4]:
from torch.utils.data import Dataset, DataLoader


class TextDataset(Dataset):
    def __init__(self, text, seq_len):
        self.text = text
        self.seq_len = seq_len
        self.data = [word_to_index[ch] for ch in text]

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, index):
        return (
            torch.tensor(self.data[index:index + self.seq_len]),
            torch.tensor(self.data[index + 1:index + self.seq_len + 1])
        )


dataset = TextDataset(text, max_seq_len)
dataloader = DataLoader(dataset, batch_size=16)
for batch in dataloader:
    print(batch)
    break


[tensor([[12, 27, 29, 82, 70],
        [27, 29, 82, 70, 27],
        [29, 82, 70, 27, 38],
        [82, 70, 27, 38, 30]]), tensor([[27, 29, 82, 70, 27],
        [29, 82, 70, 27, 38],
        [82, 70, 27, 38, 30],
        [70, 27, 38, 30, 98]])]


In [7]:

model = GPT(vocab_size, max_seq_len)
optimizer = torch.optim.Adam(model.parameters(), lr=6e-4)
criterion = nn.CrossEntropyLoss()

for _ in range(100):
    for inputs, targets in dataloader:
        batch_size, seq_len = inputs.shape
        mask = torch.tril(torch.ones(seq_len, seq_len))
        outputs = model(inputs, mask)
        loss = criterion(outputs.view(-1, vocab_size), targets.view(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(loss.item())


4.489621162414551
2.5932393074035645
0.9829242825508118
0.4140385091304779
0.2721656262874603
0.19735519587993622
0.11753450334072113
0.07806004583835602
0.058620523661375046
0.04605299234390259
0.03811350464820862
0.03126702830195427
0.028026681393384933
0.02419964037835598
0.02200237289071083
0.019454799592494965
0.01801856979727745
0.016307037323713303
0.015032082796096802
0.013773207552731037
0.012872308492660522
0.011870305985212326
0.011163143441081047
0.010427972301840782
0.009746362455189228
0.009206063114106655
0.008622294291853905
0.008129497990012169
0.007685616612434387
0.00725636025890708
0.006861499510705471
0.006572688464075327
0.006194646470248699
0.005967495031654835
0.0056011853739619255
0.005372258834540844
0.005097141955047846
0.0049031758680939674
0.00465987017378211
0.004508025012910366
0.004283362068235874
0.004137308802455664
0.003951283637434244
0.0038358450401574373
0.0036539845168590546
0.003564706537872553
0.0033875044900923967
0.0032968628220260143
0.003148

In [13]:
def generate_text(model, start_text, max_len=25, temperature=0.7, repetition_penalty=2.0):
    model.eval()
    with torch.no_grad():
        inputs = torch.tensor([word_to_index[ch] for ch in start_text])
        inputs = inputs.unsqueeze(0)
        generated_tokens = []

        for _ in range(max_len):
            batch_size, seq_len = inputs.shape
            mask = torch.tril(torch.ones(seq_len, seq_len))
            outputs = model(inputs, mask)

            next_token_logits = outputs[:, -1, :]

            for token in set(generated_tokens):
                next_token_logits[0, token] /= repetition_penalty

            next_token_logits = next_token_logits / temperature

            next_token = torch.multinomial(torch.softmax(next_token_logits, dim=-1), num_samples=1)
            generated_tokens.append(next_token.item())
            inputs = torch.cat([inputs, next_token], dim=1)

        return ''.join([index_to_word[i] for i in generated_tokens])

print(generate_text(model, "臣密言："))


臣以险衅，夙遭闵凶。
生孩六月，门衰祚薄，晚有儿息
